# Recovery Lab: reproducible final POC
TVS Credit EPIC 8. Run all cells in order from this folder using Python 3.12 and the supplied requirements. The original workbook is unchanged. This notebook uses its audited canonical records and frozen model. It reproduces inference and final evaluation; it never retrains or retunes on final outcomes.

The packaged `reproducibility/phase3/src` contains the original training workflow. Earlier phase reports describe development decisions. Original workbook provenance is in Phase 1. Keep the canonical dataset within the competition’s permitted sharing scope.

In [1]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
ROOT = Path.cwd()
assert (ROOT/'reproducibility').exists(), 'Open the notebook from its submission folder'
R = ROOT/'reproducibility'
sys.path.insert(0,str(R/'phase9/src'))
def read(p): return json.loads(p.read_text(encoding='utf-8'))
def records(p): return [json.loads(x) for x in p.read_text(encoding='utf-8').splitlines()]
canonical_path=R/'phase1/data/canonical_records.jsonl'
assert hashlib.sha256(canonical_path.read_bytes()).hexdigest() == '34a903573d80aed79b58da9bc59a11e3764c2f7ada5ac030e99536b8efe50be0'
data=pd.DataFrame(records(canonical_path))
print('Canonical rows:',len(data), 'fields:',len(data.columns))
print(data['asset_fuel_type'].value_counts().to_string())

Canonical rows: 15000 fields: 49
asset_fuel_type
Petrol    14694
EV          306


## Audit and feature meaning
The audit retained 15,000 rows, matched 615,000 source cells and found no nulls or duplicate rows. Eight derived fields bring the canonical schema to 49 fields. CIBIL -1 appears 8,850 times and has unresolved meaning. Salary frequency and existing obligations are missing definitions. Agreement duration is not manufacture age. `Data_Dictionary.md` includes all 49 fields.

Only vehicle/location attributes and agreement duration enter valuation. IDs, observed sale, yard duration, unresolved condition codes and borrower credit variables do not enter the model. The sold-only cohort cannot identify default probability or validated long-horizon values.

In [2]:
from frozen_valuation import FEATURES
assert len(data)==15000 and len(FEATURES)==9
assert not {'sold_price','yard_duration','agreement_id','cibil_score'} & set(FEATURES)
print('Valuation inputs:', '\n'.join(FEATURES))
manifest=pd.DataFrame(records(R/'phase2/data/operational_split_manifest.jsonl'))
print(manifest['role'].value_counts().to_string())
development=set(manifest.loc[manifest.role.isin(['train','tuning','calibration']),'agreement_id'])
final_ids=sorted(manifest.loc[manifest.role=='final_test','agreement_id'])
assert len(final_ids)==1244 and not development.intersection(final_ids)

Valuation inputs: customer_branch
customer_region
customer_state
pincode_tier
asset_variant
asset_model
asset_fuel_type
asset_cost_at_disbursal
months_since_agreement_at_seizure
role
train          9767
excluded       2830
final_test     1244
calibration     583
tuning          576


## Frozen baseline, model selection and calibration
Training labels precede 1 January 2026. Tuning labels precede 1 March and calibration labels precede 1 May. Six CatBoost trials selected depth 6, learning rate 0.05, L2 3 and 474 trees using 576 tuning cases. MultiQuantile estimates 0.1/0.5/0.9 quantiles. B0 uses a training-only median sale-to-cost ratio with hierarchical model/duration fallbacks.

We recompute the original calibration correction using only the 583 calibration cases. This is a reproducibility check; no new model or interval version is published. Conformal assumptions and sold-only sampling limit guarantees.

In [3]:
from calibrated import CalibratedValuator, fit_correction
from frozen_baseline import MedianRatioBaseline
model=CalibratedValuator(R/'phase9/models')
baseline=MedianRatioBaseline.load(R/'phase9/models/B0_median_ratio.json')
indexed=data.set_index('agreement_id')
cal_ids=sorted(manifest.loc[manifest.role=='calibration','agreement_id'])
cal=indexed.loc[cal_ids]
print('Calibration metadata:',model.calibration)
# The source target field is resolved from the packaged data dictionary.
dictionary=read(R/'phase9/configs/data_dictionary.json')
print([(d['canonical_name'],d['role']) for d in dictionary if 'target' in str(d['role']).lower()])
target=next(d['canonical_name'] for d in dictionary if d['role']=='target')
recomputed,_=fit_correction(cal[target].to_numpy(),model.model.predict(cal[FEATURES]))
assert abs(recomputed['correction_inr']-model.calibration['correction_inr'])<1e-7
print('Reproduced calibration correction:',recomputed['correction_inr'])


Calibration metadata: {'n': 583, 'alpha': 0.2, 'target_coverage': 0.8, 'rank_one_based': 468, 'correction_inr': 1859.2092150430399, 'version': 'phase4-operational-80-v1', 'model_sha256': '1b0c5949af11932f3ea693fee55a91aa68bc292d524e42f45151b5acd707d0d3', 'available_from': '2026-05-01', 'fit_role': 'operational.calibration', 'method': 'nonnegative conformal quantile expansion', 'calibration_scope': 'pooled observed sold assets', 'coverage_assumption': 'Exchangeability not established; no guaranteed temporal or per-segment coverage', 'lower_upper_are_not_individually_calibrated_quantiles': True}
[('sale_amount', 'target')]
Reproduced calibration correction: 1859.2092150430399


In [4]:
# Compare frozen predictions directly with the archived final predictions.
test=indexed.loc[final_ids]
pred=model.predict(test[FEATURES], '2026-09-06')
base=baseline.predict(test[FEATURES])['prediction'].to_numpy()
archived=pd.DataFrame(records(R/'phase9/data/operational_final_predictions.jsonl')).set_index('agreement_id').loc[final_ids]
np.testing.assert_allclose(pred[:,1],archived.median_inr,rtol=0,atol=1e-7)
np.testing.assert_allclose(pred[:,0],archived.calibrated_lower_inr,rtol=0,atol=1e-7)
np.testing.assert_allclose(base,archived.baseline_inr,rtol=0,atol=1e-7)
y=archived.actual_sale_inr.to_numpy()
mae=np.abs(pred[:,1]-y).mean(); bmae=np.abs(base-y).mean()
coverage=((y>=pred[:,0]) & (y<=pred[:,2])).mean()
metrics={'cases':len(y),'model_mae_inr':mae,'baseline_mae_inr':bmae,'mae_reduction_percent':100*(1-mae/bmae),'coverage':coverage,'mean_width_inr':(pred[:,2]-pred[:,0]).mean()}
print(json.dumps(metrics,indent=2))
assert abs(mae-7051.863755005328)<1e-6 and abs(coverage-1001/1244)<1e-12

{
  "cases": 1244,
  "model_mae_inr": 7051.863755005328,
  "baseline_mae_inr": 8342.911643380034,
  "mae_reduction_percent": 15.474787982432147,
  "coverage": 0.8046623794212219,
  "mean_width_inr": 23145.883082799446
}

## Final evidence and segment limitations
The pooled interval covers 1,001/1,244 outcomes (80.5%). Mean width is INR 23,146. EV coverage is only 22/36 (61.1%) with MAE INR 13,509 and roughly 0.4% improvement over baseline. This prevents a broad claim of subgroup reliability. The 1,835-case retrospective diagnostic includes all 1,244 operational cases. It is not a second independent replication. Later seizures lack complete outcome follow-up. Descriptive full-file profiling occurred before model development.

In [5]:
for fuel in ['Petrol','EV']:
    mask=test.asset_fuel_type.to_numpy()==fuel
    covered=((y[mask]>=pred[mask,0]) & (y[mask]<=pred[mask,2]))
    print(fuel, 'n=',int(mask.sum()),'MAE=',float(np.abs(pred[mask,1]-y[mask]).mean()),'covered=',int(covered.sum()),'coverage=',float(covered.mean()))
print('Prespecified diagnostic examples:')
print(json.dumps(read(R/'phase9/reports/diagnostic_examples.json'),indent=2))

Petrol n= 1208 MAE= 6859.438452712487 covered= 979 coverage= 0.8104304635761589
EV n= 36 MAE= 13508.801676387367 covered= 22 coverage= 0.6111111111111112
Prespecified diagnostic examples:
[
  {
    "selection": "Nearest median absolute error",
    "case": {
      "agreement_id": "ASSET_1004",
      "protocol": "operational",
      "role": "final_test",
      "seizure_date": "2026-05-21",
      "sold_date": "2026-06-25",
      "seizure_month": "2026-05",
      "sale_month": "2026-06",
      "asset_fuel_type": "Petrol",
      "asset_model": "MOPEDS",
      "customer_region": "AP",
      "agreement_duration_bin": "6",
      "actual_sale_inr": 34000.0,
      "baseline_inr": 39659.22032413491,
      "baseline_reference_level": "model_duration",
      "base_lower_inr": 30550.78600432664,
      "median_inr": 39308.497629125384,
      "base_upper_inr": 47100.27499616612,
      "calibrated_lower_inr": 28691.5767892836,
      "calibrated_upper_inr": 48959.48421120916,
      "model_available_at_s

## Business simulation and explicit assumptions
The separate fictional reference uses cost INR 100,000, downside INR 70,000, median INR 85,000 and verified-by-assumption income/obligations INR 30,000/5,000. The frozen simulator evaluates 35 offers. Base yields INR 80,000 for 18 months, adverse INR 70,000 for 12 months and severe no offer. These values are policy scenarios, not demonstrated profit or credit approvals.

The next cell executes the packaged API calculation through a local in-process client. No running web server is needed. An isolated temporary audit directory prevents changes to an existing user journal.

In [6]:
import tempfile, os
runtime=tempfile.TemporaryDirectory(prefix='recovery-notebook-')
os.environ['TVS_RUNTIME_DIR']=runtime.name
sys.path.insert(0,str(ROOT/'local_app/api'))
from fastapi.testclient import TestClient
from service import app
ref={'asset_id':'DEMO-A','valuation_date':'2026-09-06','downside_anchor_inr':70000,'median_anchor_inr':85000}
with TestClient(app) as client:
    for scenario,expected in [('base',80000),('adverse',70000),('severe',None)]:
        payload={'request_id':'notebook-'+scenario,'reference':ref,'scenario':scenario,'asset_reference_cost_inr':100000,'verified_monthly_income_inr':30000,'existing_monthly_obligations_inr':5000,'income_verified':True,'obligations_verified':True}
        response=client.post('/v1/lending-simulations',json=payload)
        assert response.status_code==200,response.text
        result=response.json(); offer=result['recommended_offer']
        assert (offer['principal_inr'] if offer else None)==expected
        print(scenario,offer,'feasible candidates',result['feasible_candidate_count'])
    assets=client.get('/v1/demo-portfolio').json()['assets']
    for scenario,expected in [('base',62000),('adverse',89380),('severe',121880)]:
        loans=[{'reference':{k:a[k] for k in ['asset_id','valuation_date','downside_anchor_inr','median_anchor_inr']},**{k:a[k] for k in ['existing_principal_inr','annual_nominal_rate','remaining_term_months']}} for a in assets]
        response=client.post('/v1/portfolio-scenarios',json={'scenario':scenario,'default_month':0,'assets':loans})
        assert response.status_code==200,response.text
        result=response.json();assert result['downside_shortfall_inr']==expected
        print(scenario,'book shortfall',expected,'ratio',result['exposure_weighted_shortfall_ratio'])
runtime.cleanup()

C:\Users\isash\Documents\Codex\2026-09-06\now\work\phase7_deps\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


base {'candidate_id': 'L80.0_T18', 'ltv': 0.8, 'principal_inr': 80000.0, 'tenure_months': 18, 'base_annual_nominal_rate': 0.18, 'annual_policy_premium': 0.03, 'annual_nominal_rate': 0.21, 'emi_inr': 5219.59, 'maximum_instalment_inr': 5219.66, 'total_contractual_interest_inr': 13952.69, 'foir': 0.3406553333333333, 'feasible': True, 'constraint_failures': [], 'selection_reasons': ['Highest feasible principal in the configured grid', 'Lowest contractual interest among offers with that principal', 'Shorter tenure breaks remaining ties'], 'scheduled_payments': [{'month': 0, 'payment': 0.0, 'interest': 0.0, 'principal_paid': 0.0, 'balance': 80000.0}, {'month': 1, 'payment': 5219.59, 'interest': 1400.0, 'principal_paid': 3819.59, 'balance': 76180.41}, {'month': 2, 'payment': 5219.59, 'interest': 1333.16, 'principal_paid': 3886.43, 'balance': 72293.98}, {'month': 3, 'payment': 5219.59, 'interest': 1265.14, 'principal_paid': 3954.45, 'balance': 68339.53}, {'month': 4, 'payment': 5219.59, 'inter

## Model documentation and next evidence
Read `Model_Card.md`, `Data_Dictionary.md`, `Open_Definitions.md` and `Demo_Script.md`. The model remains frozen after final outcomes were inspected. Any model update needs a fresh forward evaluation. Before production, obtain prospective and unsold outcomes, verified input timestamps, fuller recovery costs and EV validation. Add identity/access management, approved policy and operational ownership.

References: [CatBoost objectives](https://catboost.ai/docs/en/concepts/loss-functions-regression); [Romano et al., Conformalized Quantile Regression (2019)](https://arxiv.org/abs/1905.03222). This POC implements a fixed correction, not adaptive online calibration. Library authors provide CatBoost, FastAPI and React. The project supplies the integrated time controls, policy accounting, scenarios and evidence workflow.